In [17]:
# %%
"""
Dedalus script simulating 2D horizontally-periodic Rayleigh-Benard convection.
This script demonstrates solving a 2D Cartesian initial value problem. It can
be ran serially or in parallel, and uses the built-in analysis framework to save
data snapshots to HDF5 files. The `plot_snapshots.py` script can be used to
produce plots from the saved data. It should take about 5 cpu-minutes to run.

For incompressible hydro with two boundaries, we need two tau terms for each the
velocity and buoyancy. Here we choose to use a first-order formulation, putting
one tau term each on auxiliary first-order gradient variables and the others in
the PDE, and lifting them all to the first derivative basis. This formulation puts
a tau term in the divergence constraint, as required for this geometry.

To run and plot using e.g. 4 processes:
    $ mpiexec -n 4 python3 rayleigh_benard.py
    $ mpiexec -n 4 python3 plot_snapshots.py snapshots/*.h5
"""


# %%
import numpy as np
import dedalus.public as d3
import logging
logger = logging.getLogger(__name__)
import copy
import h5py
import numpy as np
import matplotlib
import re

import matplotlib.pyplot as plt
from dedalus.extras import plot_tools

from mpl_toolkits.mplot3d import Axes3D
from matplotlib.colors import Normalize

import os
from os import listdir

save_dir= "/scratch/zb2113/DedalusData/2D/justforplot2"

#if not os.path.exists(save_dir):
#    os.mkdir(save_dir)

# %%
# Parameters
Lx, Lz = 4,1
Nx, Nz = 256, 64
Ra_M = 1e6
D_0 = 0
D_H = 1/3
M_0 = 0
M_H = -1
N_s2=4/3
Qrad=0.0028

Prandtl = 1
dealias = 3/2
stop_sim_time = 50
timestepper = d3.RK222
max_timestep = 0.125
dtype = np.float64
analysison = False
savefreq = 5

# The code check for the existence of a MRBC2D_param.py and import it. This allows yo update the simulations parameters

# if ( os.path.isfile('./MRBC2D_param.py')):
#     from MRBC2D_param import *

print(Ra_M)
# %%
# Bases
coords = d3.CartesianCoordinates('x','z')
dist = d3.Distributor(coords, dtype=dtype)
xbasis = d3.RealFourier(coords['x'], size=Nx, bounds=(0, Lx), dealias=dealias)
zbasis = d3.ChebyshevT(coords['z'], size=Nz, bounds=(0, Lz), dealias=dealias)

# Fields
p = dist.Field(name='p', bases=(xbasis,zbasis))
D = dist.Field(name='D', bases=(xbasis,zbasis))
M = dist.Field(name='M', bases=(xbasis,zbasis))
u = dist.VectorField(coords, name='u', bases=(xbasis,zbasis))
Z = dist.Field(name='Z', bases=zbasis)
T = dist.Field(name='T', bases=(xbasis,zbasis))
C = dist.Field(name='C', bases=(xbasis,zbasis))
Qr = dist.Field(name='Q', bases=(xbasis,zbasis))
Nsz = dist.Field(name='Nsz', bases=(xbasis,zbasis))

tau_p = dist.Field(name='tau_p')
tau_B1 = dist.Field(name='tau_B1', bases=xbasis)
tau_B2 = dist.Field(name='tau_B2', bases=xbasis)
tau_D1 = dist.Field(name='tau_D1', bases=xbasis)
tau_D2 = dist.Field(name='tau_D2', bases=xbasis)
tau_M1 = dist.Field(name='tau_M1', bases=xbasis)
tau_M2 = dist.Field(name='tau_M2', bases=xbasis)
tau_u1 = dist.VectorField(coords,name='tau_u1', bases=xbasis)
tau_u2 = dist.VectorField(coords,name='tau_u2', bases=xbasis)
tau_u3 = dist.Field(name='tau_u3', bases=xbasis)
tau_u4 = dist.Field(name='tau_u4', bases=xbasis)
tau_T1 = dist.Field(name='tau_t1', bases=xbasis)
tau_T2 = dist.Field(name='tau_t2', bases=xbasis)
tau_C1 = dist.Field(name='tau_c1', bases=xbasis)
tau_C2 = dist.Field(name='tau_c2', bases=xbasis)

# Substitutions    
#Kuo_Bretherton Equilibrium
kappa = (Ra_M * Prandtl/((M_0-M_H)*Lz**3))**(-1/2)
nu = (Ra_M / (Prandtl*(M_0-M_H)*Lz**3))**(-1/2)
print('kappa',kappa)
print('nu',nu)
print('RaM',Ra_M)
print('Qrad',Qrad)


x,z = dist.local_grids(xbasis,zbasis)
Z['g']=z
Qr['g']=np.sin(np.pi*z/Lz)
Nsz['g']=N_s2*z

ex,ez = coords.unit_vector_fields(dist)
lift_basis = zbasis.derivative_basis(1)
lift = lambda A: d3.Lift(A, lift_basis, -1)

B_op = (np.absolute(D - M - Nsz)+ M + D - Nsz)/2
lq = B_op/2 + np.absolute(B_op)/2



Max = lambda A,B: (abs(A-N_s2*Z-B)+A-N_s2*Z+B)/2
eva = lambda A: A.evaluate()

dz= lambda A: d3.Differentiate(A, coords['z'])
dx= lambda A: d3.Differentiate(A, coords['x'])


ux=u@ex
uz=u@ez
dxux=dx(ux)
dzux=dz(ux)
dxuz=dx(uz)
dzuz=dz(uz)

dzlq = d3.Differentiate(lq, coords['z'])
dzD = d3.Differentiate(D, coords['z'])
dzM = d3.Differentiate(M, coords['z'])
dzC = d3.Differentiate(C, coords['z'])
dzT = d3.Differentiate(T, coords['z'])

grad_u = d3.grad(u) + ez* lift(tau_u1) # First-order reduction
grad_ux = grad_u@ex # First-order reduction
grad_uz = grad_u@ez # First-order reduction
grad_M = d3.grad(M) + ez*lift(tau_M1) # First-order reduction
grad_D = d3.grad(D) + ez*lift(tau_D1) # First-order reduction
grad_T = d3.grad(T) + ez*lift(tau_T1)
grad_C = d3.grad(C) + ez*lift(tau_C1)

# Problem
# First-order form: "div(f)" becomes "trace(grad_f)"
# First-order form: "lap(f)" becomes "div(grad_f)"
problem = d3.IVP([p, M, D, u, T, C, tau_p, tau_M1, tau_M2, tau_D1, tau_D2, tau_u1, tau_u2, tau_T1, tau_T2, tau_C1, tau_C2], namespace=locals())
problem.add_equation("trace(grad_u) + tau_p= 0")
problem.add_equation("dt(M) - kappa*div(grad_M) + lift(tau_M2) = - u@grad(M)-Qrad/2*Qr")
problem.add_equation("dt(D) - kappa*div(grad_D) + lift(tau_D2) = - u@grad(D)-Qrad*Qr")
problem.add_equation("dt(u) - nu*div(grad_u) + grad(p)  + lift(tau_u2) = - u@grad(u)+ B_op*ez")
problem.add_equation("dt(T) - kappa*div(grad_T) + lift(tau_T2) = - u@grad(T)")
problem.add_equation("dt(C) - kappa*div(grad_C) + lift(tau_C2) = - u@grad(C)+1")
problem.add_equation("u(z=0) = 0")
problem.add_equation("uz(z=Lz) = 0")
problem.add_equation("dz(ux)(z=Lz)=0")
problem.add_equation("M(z=0) = M_0")
problem.add_equation("D(z=0) = D_0")
problem.add_equation("M(z=Lz) = M_H")
problem.add_equation("D(z=Lz) = D_H")
problem.add_equation("T(z=0) = 0")
problem.add_equation("T(z=1) = 1")
problem.add_equation("C(z=0) = 0")
problem.add_equation("dz(C)(z=Lz) = 0")
problem.add_equation("integ(p) = 0") # Pressure gauge

# %%
# Solver
solver = problem.build_solver(timestepper)
solver.stop_sim_time = stop_sim_time


# %%
# Initial condition
D.fill_random('g', seed=42, distribution='normal', scale=1e-3) # Random noise
D['g'] *= z * (Lz - z) # Damp noise at walls
D['g'] += (D_H-D_0)*z+D_0 # Add linear background
M.fill_random('g', seed=28, distribution='normal', scale=1e-3) # Random noise
M['g'] *= z * (Lz - z) # Damp noise at walls
M['g'] += (M_H-M_0)*z +M_0 # Add linear background



# %%
# Analysis
analysis = solver.evaluator.add_file_handler('./analysis', sim_dt=0.25, max_writes=1)
analysis.add_task(D, name='D')
analysis.add_task(M, name='M')
analysis.add_task(C, name='C')
analysis.add_task(T, name='T')
analysis.add_task(B_op, name='B')
analysis.add_task(uz, name='uz')
analysis.add_task(uz*C, name='C flux')
analysis.add_task(uz*B_op, name='B flux')
analysis.add_task(uz*T, name='T flux')
analysis.add_task(d3.Average(C, coords['x']), name='horizontal avg C')
analysis.add_task(d3.Average(uz*C, coords['x']), name='horizontal avg C flux')



# %%
# CFL
CFL = d3.CFL(solver, initial_dt=0.1, cadence=1, safety=0.3, threshold=0.05,
             max_change=1.1, min_change=0.25, max_dt=max_timestep)
CFL.add_velocity(u)

# %%
# Flow properties
flow = d3.GlobalFlowProperty(solver, cadence=10)
flow.add_property(np.sqrt(u@u)/nu, name='Re')


# %%
# Main loop
startup_iter = 10
try:
    logger.info('Starting main loop')
    while solver.proceed:
        timestep = CFL.compute_timestep()
        solver.step(timestep)
        if (solver.iteration-1) % 10 == 0:
            max_Re = flow.max('Re')
            logger.info('Iteration=%i, Time=%e, dt=%e, max(Re)=%f' %(solver.iteration, solver.sim_time, timestep, max_Re))
except:
    logger.error('Exception raised, triggering end of main loop.')
    raise
finally:
    solver.log_stats()



1000000.0
kappa 0.001
nu 0.001
RaM 1000000.0
Qrad 0.0028
2024-10-06 20:22:23,607 subsystems 0/1 INFO :: Building subproblem matrices 1/128 (~1%) Elapsed: 0s, Remaining: 20s, Rate: 6.3e+00/s
2024-10-06 20:22:24,779 subsystems 0/1 INFO :: Building subproblem matrices 13/128 (~10%) Elapsed: 1s, Remaining: 12s, Rate: 9.8e+00/s
2024-10-06 20:22:26,043 subsystems 0/1 INFO :: Building subproblem matrices 26/128 (~20%) Elapsed: 3s, Remaining: 10s, Rate: 1.0e+01/s
2024-10-06 20:22:27,310 subsystems 0/1 INFO :: Building subproblem matrices 39/128 (~30%) Elapsed: 4s, Remaining: 9s, Rate: 1.0e+01/s
2024-10-06 20:22:28,576 subsystems 0/1 INFO :: Building subproblem matrices 52/128 (~41%) Elapsed: 5s, Remaining: 7s, Rate: 1.0e+01/s
2024-10-06 20:22:29,848 subsystems 0/1 INFO :: Building subproblem matrices 65/128 (~51%) Elapsed: 6s, Remaining: 6s, Rate: 1.0e+01/s
2024-10-06 20:22:31,114 subsystems 0/1 INFO :: Building subproblem matrices 78/128 (~61%) Elapsed: 8s, Remaining: 5s, Rate: 1.0e+01/s
2024

In [19]:
from lib.dedalus_Plot import Plot
folder="justforplot3"
# Plotter=Plot(save_dir="/scratch/zb2113/DedalusData/2D/"+folder,handler='analysis')
Plotter=Plot(save_dir="/home/zb2113/Research-Dedalus/2d-AR=64",handler='analysis')
Plotter.get_sim_time()
print(Plotter.sim_time)
Plotter.plot_all_snapshots('B', output_dir="/home/zb2113/Dedalus-Postanalysis/2D/"+folder,levelnum=20)
Plotter.plot_all_snapshots('C', output_dir="/home/zb2113/Dedalus-Postanalysis/2D/"+folder,levelnum=20)
# Plotter.plot_all_snapshots('M', output_dir="/home/zb2113/Dedalus-Postanalysis/2D/"+folder,levelnum=20)
# Plotter.animate('C',use_existing_pics=True,output_dir="/home/zb2113/Dedalus-Postanalysis/2D/"+folder,output_type='mp4')
# Plotter.animate('M',use_existing_pics=True,output_dir="/home/zb2113/Dedalus-Postanalysis/2D/"+folder,output_type='mp4')

['scales', 'tasks']
Scale keys: ['constant', 'iteration', 'sim_time', 'timestep', 'wall_time', 'write_number', 'x_hash_259ff677ed0cf648980d6e69fe3d2deb0dc82141', 'z_hash_2b3e6c1ad6197e7bbb577c37c9be3babe1727daf']
Task keys: ['B', 'B flux', 'C', 'C flux', 'D', 'M', 'T', 'T flux', 'horizontal avg C', 'horizontal avg C flux', 'uz']
[0.0, 0.2, 0.552, 0.794, 1.036, 1.278, 1.52, 1.762, 2.004, 2.246, 2.488, 2.73, 2.972, 3.214, 3.456, 3.698, 3.94, 4.303000000000001, 4.545000000000002, 4.787000000000003, 5.0290000000000035, 5.271000000000004, 5.513000000000005, 5.755000000000006, 5.997000000000007, 6.239000000000008, 6.481000000000009, 6.72300000000001, 6.9650000000000105, 7.207000000000011, 7.449000000000012, 7.691000000000013, 8.054000000000014, 8.296000000000015, 8.538000000000016, 8.780000000000017, 9.022000000000018, 9.264000000000019, 9.50600000000002, 9.74800000000002, 9.990000000000022, 10.232000000000022, 10.474000000000023, 10.716000000000024, 11.048846450435812, 11.230918743654524, 1